In [ ]:
!pip install -U docling pinecone beautifulsoup4 requests 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 9.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of typer-slim to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.4/451.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 97.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 3.3 MB/s eta 0:00:00
   ━━━

In [2]:
!pip uninstall -y transformers sentence-transformers peft FlagEmbedding

!pip install \
  "transformers==4.40.2" \
  "sentence-transformers==2.6.1" \
  "FlagEmbedding==1.2.10" \
  "peft==0.10.0"

Found existing installation: transformers 5.5.0
Uninstalling transformers-5.5.0:
  Successfully uninstalled transformers-5.5.0
Found existing installation: sentence-transformers 5.2.3
Uninstalling sentence-transformers-5.2.3:
  Successfully uninstalled sentence-transformers-5.2.3
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.3/141.3 kB 12.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 99.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.3/163.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.8 MB/s eta 0:00:00:00

In [3]:
import json
import os
import time
from pathlib import Path
from dotenv import load_dotenv

import numpy as np
from FlagEmbedding import BGEM3FlagModel
from pinecone import Pinecone, ServerlessSpec

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()


CHUNK_FILES = [
    "/kaggle/input/datasets/saadimam2020/fixed-size/layer1_rulebook_chunks_fixed.json",
    "/kaggle/input/datasets/saadimam2020/fixed-size/layer2_chunks_fixed.json"
]

# We reuse the same index as the hybrid experiment and separate the two
# chunking strategies with Pinecone namespaces.
# hybrid chunks live in the default ("") namespace;
# fixed-size chunks live in the "fixed-size" namespace.
INDEX_NAME = "basketball-rag-hybrid-bge"
NAMESPACE  = "fixed-size"
PINECONE_API_KEY = user_secrets.get_secret("PINECONE_API_KEY")

# BGE-M3 settings
# use_fp16=True cuts memory usage roughly in half on GPU with minimal quality loss
# On CPU, set use_fp16=False
BGE_MODEL_NAME = "BAAI/bge-m3"
USE_FP16 = False      # set False if running on CPU only


# On CPU:
#   - batch_size=4 is safe, expect ~2-5 minutes per 100 chunks
EMBED_BATCH_SIZE = 4
# Pinecone upsert batch size 
UPSERT_BATCH_SIZE = 100
# BGE-M3's dense embedding dimension
DENSE_DIMENSION = 1024

# STEP 1: Load all chunks from JSON files
def load_all_chunks(chunk_files: list[str]) -> list[dict]:
    
    # Each chunk must have: chunk_id, text, metadata
    all_chunks = []

    for filepath in chunk_files:
        if not os.path.exists(filepath):
            print(f"  [!] File not found, skipping: {filepath}")
            continue

        with open(filepath, "r", encoding="utf-8") as f:
            chunks = json.load(f)

        print(f"  Loaded {len(chunks)} chunks from: {filepath}")
        all_chunks.extend(chunks)

    print(f"\nTotal chunks loaded: {len(all_chunks)}")
    return all_chunks

# STEP 2: Generate BGE-M3 embeddings (dense + sparse in one pass)


def generate_embeddings( model: BGEM3FlagModel, texts: list[str], batch_size: int = EMBED_BATCH_SIZE
                        ) -> tuple[list[list[float]], list[dict]]:
    """
    Runs BGE-M3 on a list of texts and returns:
      - dense_vectors:  list of 1024-float lists
      - sparse_vectors: list of dicts, each with 'indices' and 'values' lists

    BGE-M3 generates both in a SINGLE forward pass — no extra BM25 encoder needed.
    The return_dense and return_sparse flags tell the model what to compute.
    """
    dense_vectors = []
    sparse_vectors = []

    total_batches = (len(texts) + batch_size - 1) // batch_size

    for batch_idx in range(0, len(texts), batch_size):
        batch_texts = texts[batch_idx : batch_idx + batch_size]
        current_batch = batch_idx // batch_size + 1

        print(f"  Encoding batch {current_batch}/{total_batches} "
              f"({len(batch_texts)} texts)...", end="\r")

        output = model.encode(
            batch_texts,
            return_dense=True,         # get 1024-dim dense vectors
            return_sparse=True,        # get learned sparse vectors (like SPLADE)
            return_colbert_vecs=False, 
            batch_size=batch_size,
            max_length=512,            
        )

        # Dense vectors: shape (batch_size, 1024)
        # Convert from numpy float32 to Python lists for Pinecone
        batch_dense = output["dense_vecs"].tolist()
        dense_vectors.extend(batch_dense)

        # Sparse vectors: BGE-M3 returns a list of dicts like:
        # {"token1_id": weight1, "token2_id": weight2, ...}
        # We need to convert to Pinecone's format: {"indices": [...], "values": [...]}
        batch_sparse_raw = output["lexical_weights"]

        for sparse_dict in batch_sparse_raw:
            # sparse_dict is a DefaultDict mapping token_id (int) -> weight (float)
            # Filter out near-zero weights to keep sparse vectors compact
            # Pinecone supports up to 1000 non-zero values per sparse vector
            filtered = {
                k: float(v)
                for k, v in sparse_dict.items()
                if float(v) > 0.01  # threshold removes truly negligible tokens
            }

            # FIX: Explicitly cast k to int here
            top_tokens = sorted(
                [(int(k), float(v)) for k, v in filtered.items()], # Cast to (int, float)
                key=lambda x: x[1], 
                reverse=True
            )[:1000]
        
            if top_tokens:
                indices, values = zip(*top_tokens)
                sparse_vectors.append({
                "indices": [int(i) for i in indices[:1000]],
                "values": [float(v) for v in values[:1000]]
                })
            else:
                sparse_vectors.append({"indices": [], "values": []})

    print(f"\n  Done encoding {len(dense_vectors)} vectors.")
    return dense_vectors, sparse_vectors


# STEP 3: Create the Pinecone hybrid index


def create_pinecone_index(pc: Pinecone, index_name: str) -> None:
    existing_indexes = [idx.name for idx in pc.list_indexes()]

    if index_name in existing_indexes:
        print(f"  Index '{index_name}' already exists — skipping creation.")
        return

    print(f"  Creating index '{index_name}'...")

    pc.create_index(
        name=index_name,
        dimension=DENSE_DIMENSION,     # 1024 for BGE-M3
        metric="dotproduct",           # REQUIRED for hybrid search — do not change
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"         # Pinecone free tier is on AWS us-east-1
        )
    )

    # Wait for index to be ready before upserting
    print("  Waiting for index to be ready...")
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(2)

    print(f"  Index '{index_name}' is ready.")

# STEP 4: Upsert all vectors to Pinecone
def upsert_to_pinecone(pc: Pinecone, index_name: str, chunks: list[dict], dense_vectors: list[list[float]], sparse_vectors: list[dict],
    batch_size: int = UPSERT_BATCH_SIZE
) -> None:
    
    index = pc.Index(index_name)
    total_batches = (len(chunks) + batch_size - 1) // batch_size
    total_upserted = 0

    print(f"\nUpserting {len(chunks)} vectors in batches of {batch_size}...")

    for batch_idx in range(0, len(chunks), batch_size):
        batch_end = min(batch_idx + batch_size, len(chunks))
        batch_chunks = chunks[batch_idx:batch_end]
        batch_dense = dense_vectors[batch_idx:batch_end]
        batch_sparse = sparse_vectors[batch_idx:batch_end]

        records = []
        for chunk, dense, sparse in zip(batch_chunks, batch_dense, batch_sparse):

            # Build the metadata dict
            # We include all fields from the chunk's metadata
            # but ensure 'text' is always present (needed for retrieval display)
            meta = dict(chunk.get("metadata", {}))
            if "text" not in meta:
                meta["text"] = chunk.get("text", "")

            # Pinecone metadata values must be: str, int, float, bool, or list of str
            # Convert any None values to empty strings to avoid upsert errors
            clean_meta = {}
            for k, v in meta.items():
                if v is None:
                    clean_meta[k] = ""
                elif isinstance(v, list):
                    # Convert list items to strings if they aren't already
                    clean_meta[k] = [str(i) for i in v]
                else:
                    clean_meta[k] = v

            record = {
                "id": chunk["chunk_id"],
                "values": dense,
                "sparse_values": sparse,
                "metadata": clean_meta
            }
            records.append(record)

        # Upsert the batch into the fixed-size namespace
        index.upsert(vectors=records, namespace=NAMESPACE)
        total_upserted += len(records)

        current_batch = batch_idx // batch_size + 1
        print(f"  Batch {current_batch}/{total_batches} — "
              f"upserted {total_upserted}/{len(chunks)} vectors")

        # Small delay to be polite to the API
        time.sleep(0.1)

    print(f"\n Upsert complete. Total vectors in index: {total_upserted}")

# STEP 5: Verify the index after upserting
def verify_index(pc: Pinecone, index_name: str) -> None:
    """
    """
    index = pc.Index(index_name)
    stats = index.describe_index_stats()

    print(f"\n{'='*60}")
    print(f"INDEX VERIFICATION — '{index_name}'")
    print(f"{'='*60}")
    print(f"Total vectors across all namespaces: {stats.total_vector_count}")
    print(f"Dimension: {stats.dimension}")
    print(f"\nPer-namespace breakdown:")
    for ns_name, ns_stats in stats.namespaces.items():
        label = f"  '{ns_name}'" if ns_name else "  '' (default / hybrid)"
        print(f"{label}: {ns_stats.vector_count} vectors")

def main():
    # Validate API key
    if not PINECONE_API_KEY:
        raise ValueError(
            "PINECONE_API_KEY not found. "
            "Create a .env file with PINECONE_API_KEY=your_key"
        )
    
    print("=" * 60)
    print("STEP 1: Loading chunks")
    print("=" * 60)
    all_chunks = load_all_chunks(CHUNK_FILES)

    if not all_chunks:
        print("No chunks found.")
        return

    # Extract just the text for embedding 
    texts = [chunk["text"] for chunk in all_chunks]

    print("\n" + "=" * 60)
    print("STEP 2: Loading BGE-M3 model")
    print("=" * 60)
    print(f"  Model: {BGE_MODEL_NAME}")
    print(f"  FP16:  {USE_FP16} ")


    model = BGEM3FlagModel(
        BGE_MODEL_NAME,
        use_fp16=USE_FP16
    )
    print("  Model loaded.")

    print("\n" + "=" * 60)
    print("STEP 3:  embeddings (dense + sparse)")
    print("=" * 60)
    print(f"  Total chunks to embed: {len(texts)}")
    print(f"  Batch size: {EMBED_BATCH_SIZE}")
  

    start_time = time.time()
    dense_vectors, sparse_vectors = generate_embeddings(model, texts)
    elapsed = time.time() - start_time
    print(f"  Embedding time: {elapsed:.1f}s  ({elapsed/len(texts)*1000:.1f}ms per chunk)")

    # Quick sanity check
    assert len(dense_vectors) == len(all_chunks), "Dense vector count mismatch"
    assert len(sparse_vectors) == len(all_chunks), "Sparse vector count mismatch"
    assert len(dense_vectors[0]) == DENSE_DIMENSION, \
        f"Dense dimension mismatch: expected {DENSE_DIMENSION}, got {len(dense_vectors[0])}"

    print(f"  Dense dimension:  {len(dense_vectors[0])} ✓")
    print(f"  Sample sparse non-zeros: {len(sparse_vectors[0]['indices'])} tokens")

    # -----------------------------------------------------------------------
    # Save embeddings locally as a checkpoint
    # This is important: if Pinecone upsert fails halfway, you don't want to
    # re-run the 2+ hour embedding step. Save first, upsert from file if needed.
    # -----------------------------------------------------------------------
    checkpoint_path = Path("corpus/embeddings_checkpoint_fixed.json")
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"\n  Saving embedding checkpoint to {checkpoint_path}...")
    checkpoint = {
        "chunk_ids": [c["chunk_id"] for c in all_chunks],
        "dense_vectors": dense_vectors,
        "sparse_vectors": sparse_vectors,
    }
    with open(checkpoint_path, "w") as f:
        json.dump(checkpoint, f)
    print("  Checkpoint saved.")

    # Create Pinecone index
    print("\n" + "=" * 60)
    print("STEP 4: Setting up Pinecone index")
    print("=" * 60)

    pc = Pinecone(api_key=PINECONE_API_KEY)
    create_pinecone_index(pc, INDEX_NAME)

    # Upsert vectors
    print("\n" + "=" * 60)
    print("STEP 5: Upserting to Pinecone")
    print("=" * 60)

    upsert_to_pinecone(pc=pc, index_name=INDEX_NAME, chunks=all_chunks, dense_vectors=dense_vectors, sparse_vectors=sparse_vectors,)
    verify_index(pc, INDEX_NAME)

    print("\n✓ Indexing complete.")
    print(f"  Index name: '{INDEX_NAME}'")
    print(f"  Namespace:  '{NAMESPACE}'")


# ---------------------------------------------------------------------------
# RESUME FROM CHECKPOINT
# If embedding was done but upsert failed, run this instead of main()
# to skip the expensive embedding step
# ---------------------------------------------------------------------------

def upsert_from_checkpoint():
    """
    Call this function instead of main() if:
    - Embeddings were already generated and saved to checkpoint
    - But Pinecone upsert failed or was interrupted
    """
    print("Loading from checkpoint...")

    chunk_files = CHUNK_FILES
    all_chunks = load_all_chunks(chunk_files)

    checkpoint_path = "corpus/embeddings_checkpoint_fixed.json"
    with open(checkpoint_path, "r") as f:
        checkpoint = json.load(f)

    dense_vectors = checkpoint["dense_vectors"]
    sparse_vectors = checkpoint["sparse_vectors"]

    print(f"Loaded {len(dense_vectors)} embeddings from checkpoint.")

    pc = Pinecone(api_key=PINECONE_API_KEY)
    create_pinecone_index(pc, INDEX_NAME)
    upsert_to_pinecone(pc, INDEX_NAME, all_chunks, dense_vectors, sparse_vectors)
    verify_index(pc, INDEX_NAME)


if __name__ == "__main__":
    main()
    # If resuming from checkpoint, comment out main() and uncomment:
    # upsert_from_checkpoint()

2026-04-05 07:05:56.262236: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775372756.517167      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775372756.588912      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775372757.120663      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775372757.120701      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775372757.120703      55 computation_placer.cc:177] computation placer alr

STEP 1: Loading chunks
  Loaded 245 chunks from: /kaggle/input/datasets/saadimam2020/fixed-size/layer1_rulebook_chunks_fixed.json
  Loaded 3006 chunks from: /kaggle/input/datasets/saadimam2020/fixed-size/layer2_chunks_fixed.json

Total chunks loaded: 3251

STEP 2: Loading BGE-M3 model
  Model: BAAI/bge-m3
  FP16:  False 


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

bm25.jpg:   0%|          | 0.00/132k [00:00<?, ?B/s]

...

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

----------using 2*GPUs----------
  Model loaded.

STEP 3:  embeddings (dense + sparse)
  Total chunks to embed: 3251
  Batch size: 4
  Encoding batch 813/813 (3 texts)...
  Done encoding 3251 vectors.
  Embedding time: 336.9s  (103.6ms per chunk)
  Dense dimension:  1024 ✓
  Sample sparse non-zeros: 92 tokens

  Saving embedding checkpoint to corpus/embeddings_checkpoint_fixed.json...
  Checkpoint saved.

STEP 4: Setting up Pinecone index
  Index 'basketball-rag-hybrid-bge' already exists — skipping creation.

STEP 5: Upserting to Pinecone

Upserting 3251 vectors in batches of 100...
  Batch 1/33 — upserted 100/3251 vectors
  Batch 2/33 — upserted 200/3251 vectors
  Batch 3/33 — upserted 300/3251 vectors
  Batch 4/33 — upserted 400/3251 vectors
  Batch 5/33 — upserted 500/3251 vectors
  Batch 6/33 — upserted 600/3251 vectors
  Batch 7/33 — upserted 700/3251 vectors
  Batch 8/33 — upserted 800/3251 vectors
  Batch 9/33 — upserted 900/3251 vectors
  Batch 10/33 — upserted 1000/3251 vecto